# RQ2: Are there significant differences in developers’ perceptions of security practices across groups with different experience levels, professional roles, and organizational types? 

## Pré-processing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from cliffs_delta import cliffs_delta

In [2]:
dados_path = Path('data')

df_o = pd.read_excel(dados_path / 'qualitative_responses_ptbr.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

print(len(df))

print(df.columns.tolist())

95
['ID', 'Carimbo de data/hora', 'Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.', 'Qual é a sua idade?', 'Em qual estado do Brasil você mora atualmente?', 'Qual é o seu gênero?', 'Qual é o seu nível de educação formal concluída?', 'Qual seu modelo de trabalho?', 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?', 'Por favor, indique a senioridade da posição que ocupa.', 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?', 'Quantos anos de experiência você tem em funções relacionadas a Segurança de 

In [3]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?']
          == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

['Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software'
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
(83, 103)


In [4]:
df.isna().sum()

ID                                                                                                                                                                                                                                                                                                                                                                                                                                                              0
Carimbo de data/hora                                                                                                                                                                                                                                                                                                                                                                                                                                            0
Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro

In [5]:
df.shape

(83, 103)

Analises

In [6]:
categories = {
    "3rd Party Vulnerability Scanning": [
        "Dependabot", "PHP Security Checker", "Snyk", "Clair", "Blackduck",
        "Retire.JS", "SRC:CLR", "Anchore", "Node Security Platform", "Trivy",
        "Sonatype", "Whitesource"
    ],
    "Static Application Security Testing": [  # SAST
        "CodeQL", "CheckStyle", "Coverity", "SonarQube", "Bandit", "PHPStan",
        "LGTM", "Brakeman", "ESLint Security", "Dart Analyzer", "DevSkim"
    ],
    "Dynamic Application Security Testing": [  # DAST
        "Qark", "Checkmarx", "CFN NAG"
    ],
    "Secure Configuration/Hardening": [
        "Chef", "Ansible", "Terraform", "Puppet", "Phan", "Seccomp",
        "CloudCustodian", "Turbot"
    ],
    "Compliance/Hardening Checks": [
        "Bane", "Ansible-Lint", "Blackbox", "Puppet-Lint", "InSpec",
        "Kube Audit", "Foodcritic", "Cookstyle", "conftest"
    ],
    "Secrets Management": [
        "Pinterest Knox", "GitLeaks", "CyberArk Conjur",
        "ThoughtWorks Talisman", "Berglas", "Docker Secrets", "Trufflehog"
    ]
}

In [ ]:
tool_cols = [
    col for col in df.columns
    if "categorias de ferramentas" in str(col).lower()
]

print(f"\n{len(tool_cols)} columns found for the question about tools:")
for c in tool_cols:
    print("-", c)

tools_df = df[tool_cols].copy()


6 colunas encontradas para a pergunta das ferramentas:
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [3rd Party Vulnerability Scanning Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Static Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere mar

In [8]:
tools_df = tools_df.applymap(
    lambda x: x.strip() if isinstance(x, str) else x
)

C:\Users\reset\AppData\Local\Temp\ipykernel_1276\889441433.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tools_df = tools_df.applymap(


In [9]:
tools_df.insert(0, "Respondente", range(1, len(tools_df) + 1))

In [10]:
likert_scale = [
    "Diariamente",
    "Semanalmente",
    "Mensalmente",
    "Anualmente",
    "Não Aplicável",
]

In [11]:
result = {}

for category in tools_df.columns[1:]:  # ignores the "Respondente" column
    result[category] = {}
    for option in likert_scale:
        # Selects all rows where the respondent chose this option
        participants = tools_df[tools_df[category] == option][[
            "Respondente", category]]

        # Stores the individual responses
        result[category][option] = participants


In [12]:
example_category = list(result.keys())[2]
print(f"\nExample — Category: {example_category}")
for option, responses in result[example_category].items():
    print(f"\nOption: {option}")
    print(responses)


Example — Category: Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]

Option: Diariamente
    Respondente  \
4             4   
5             5   
24           21   
28           24   
32           28   
33           29   
34           30   
37           33   
54           47   
63           56   
75           66   
77           68   
80           71   
84           75   
91           81   
92           82   

   Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]  
4                                         Diariamente                                          

In [ ]:
results_path = Path("results/RQ2")

with pd.ExcelWriter(results_path / "tools_usage_frequency_responses.xlsx") as writer:
    for i, (category, responses_by_option) in enumerate(result.items()):
        frames = []
        for option, responses in responses_by_option.items():
            temp = responses.copy()
            temp["Opção"] = option
            frames.append(temp)
        category_df = pd.concat(frames, ignore_index=True)
        # ensures uniqueness and respects the 31-character limit
        sheet_name = f"{category[:27]}_{i}"
        category_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nSaved File: {results_path}/tools_usage_frequency_responses.xlsx")



Arquivo salvo: results\RQ2/tools_usage_frequency_responses.xlsx


## 'Not applicable' category


In [14]:
# List of survey columns related to security practices
security_practices_survey_cols = [
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Revisão de Código]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Modelagem de Ameaças]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Identificação de Casos de Abuso/Mau Uso]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Análise de Riscos Arquiteturais]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Penetração]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Riscos]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Vulnerabilidades]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Segurança Manuais/Automatizados]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Execução de Testes Fuzz]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Engenharia de Requisitos de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Codificação Segura]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Resposta Emergencial ao Código]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Configurações e Gerenciamento de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Monitoramento de Eventos de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Auditorias de Segurança Externas]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Segurança da Cadeia de Suprimentos de Software]'
]

In [ ]:
not_applicable_summary = []

for column in security_practices_survey_cols:
    # Count occurrences of 'Não Aplicável'
    not_applicable_count = df[column].value_counts().get('Não Aplicável', 0)

    # Count total valid responses (excluding NaN for percentage calculation)
    total_valid_responses = df[column].dropna().shape[0]

    if total_valid_responses > 0:
        not_applicable_percentage = (
            not_applicable_count / total_valid_responses) * 100
    else:
        not_applicable_percentage = 0

    not_applicable_summary.append({
        'Coluna': column,
        'Contagem Não Aplicável': not_applicable_count,
        'Porcentagem Não Aplicável': round(not_applicable_percentage, 2)
    })

df_not_applicable_summary = pd.DataFrame(not_applicable_summary)

print("DataFrame with the summary of 'Not Applicable' by column:")
display(df_not_applicable_summary)


DataFrame com o resumo de 'Não Aplicável' por coluna:


,Coluna,Contagem Não Aplicável,Porcentagem Não Aplicável
0,"Com base em sua experiência, como você avaliar...",11,13.25
1,"Com base em sua experiência, como você avaliar...",20,24.10
2,"Com base em sua experiência, como você avaliar...",18,21.69
3,"Com base em sua experiência, como você avaliar...",14,16.87
4,"Com base em sua experiência, como você avaliar...",15,18.07
...,...,...,...
59,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
60,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
61,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
62,Qual a sua opinião quanto a frase a seguir apl...,0,0.00


In [ ]:
# Identify 'ease of use' and 'effectiveness' columns
ease_of_use_cols = [col for col in df_not_applicable_summary['Coluna']
                       if 'facilidade de uso' in col.lower()]
effectiveness_cols = [col for col in df_not_applicable_summary['Coluna']
                          if 'efetividade das seguintes práticas' in col.lower()]

# Filter the summary DataFrame to get the percentages
df_ease_of_use = df_not_applicable_summary[df_not_applicable_summary['Coluna'].isin(
    ease_of_use_cols)]
df_effectiveness = df_not_applicable_summary[df_not_applicable_summary['Coluna'].isin(
    effectiveness_cols)]

# Calculate the mean percentages for each group
ease_of_use_mean = df_ease_of_use['Porcentagem Não Aplicável'].mean()
effectiveness_mean = df_effectiveness['Porcentagem Não Aplicável'].mean()

# Create a new DataFrame with the results
df_applicability_summary = pd.DataFrame({
    'Atributo': ['Facilidade de Uso', 'Efetividade'],
    'Média Porcentagem Não Aplicável': [round(ease_of_use_mean, 2), round(effectiveness_mean, 2)]
})

print("DataFrame summarizing the average of 'Not Applicable' by attribute:")
display(df_applicability_summary)

DataFrame com o resumo da média de 'Não Aplicável' por atributo:


,Atributo,Média Porcentagem Não Aplicável
0,Facilidade de Uso,22.74
1,Efetividade,19.35


In [17]:
from pathlib import Path
dados_path = Path('data')

df_o = pd.read_excel(
    dados_path / 'survey_all_responses_ptbr.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

In [18]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?']
          == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

['Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software'
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
(83, 102)


In [19]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna().sum())

df = df.drop(df[df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna()].index, axis=0)

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna().sum())

0
0


In [20]:
pratices = ['[Revisão de Código]',
            '[Modelagem de Ameaças]',
            '[Identificação de Casos de Abuso/Mau Uso]',
            '[Análise de Riscos Arquiteturais]',
            '[Testes de Penetração]',
            '[Gerenciamento de Riscos]',
            '[Gerenciamento de Vulnerabilidades]',
            '[Testes de Segurança Manuais/Automatizados]',
            '[Execução de Testes Fuzz]',
            '[Engenharia de Requisitos de Segurança]',
            '[Codificação Segura]',
            '[Resposta Emergencial ao Código]',
            '[Configurações e Gerenciamento de Segurança]',
            '[Monitoramento de Eventos de Segurança]',
            '[Auditorias de Segurança Externas]',
            '[Segurança da Cadeia de Suprimentos de Software]']

In [21]:
frequency_cols = ['Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
                      ]

In [22]:
ease_of_use_cols = ['Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]'
                      ]

In [23]:
effectiveness_cols = ['Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
                       ]

In [24]:
essentiality_cols = ['Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Revisão de Código]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Modelagem de Ameaças]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Identificação de Casos de Abuso/Mau Uso]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Análise de Riscos Arquiteturais]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Penetração]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Riscos]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Vulnerabilidades]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Segurança Manuais/Automatizados]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Execução de Testes Fuzz]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Engenharia de Requisitos de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Codificação Segura]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Resposta Emergencial ao Código]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Configurações e Gerenciamento de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Monitoramento de Eventos de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Auditorias de Segurança Externas]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Segurança da Cadeia de Suprimentos de Software]']

In [25]:
frequency_map = {
    'Não aplicável': 0,
    'Anualmente': 1,
    'Mensalmente': 2,
    'Semanalmente': 3,
    'Diariamente': 4,
}

effectiveness_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

ease_of_use_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

essentiality_map = {
    'Discordo Totalmente': 1,
    'Discordo': 2,
    'Neutro': 3,
    'Concordo': 4,
    'Concordo Totalmente': 5,
}

frequency_map = {
    'Não aplicável': 0,
    'Anualmente': 1,
    'Mensalmente': 2,
    'Semanalmente': 3,
    'Diariamente': 4,
}

effectiveness_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

ease_of_use_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

essentiality_map = {
    'Discordo Totalmente': 1,
    'Discordo': 2,
    'Neutro': 3,
    'Concordo': 4,
    'Concordo Totalmente': 5,
}

frequency_map_english = {
    "Not applicable": 0,
    "Annually": 1,
    "Monthly": 2,
    "Weekly": 3,
    "Daily": 4,
}

effectiveness_map_english = {
    "Not applicable": 0,
    "Very Low": 1,
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5,
}

ease_of_use_map_english = {
    "Not applicable": 0,
    "Very Low": 1,
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5,
}

essentiality_map_english = {
    "Strongly Disagree": 1,
    "Disagree": 2,
    "Neutral": 3,
    "Agree": 4,
    "Strongly Agree": 5,
}

In [26]:
def generate_description_table(df: pd.DataFrame, order: list = None) -> pd.DataFrame:
    """
    Generates a frequency count table ('Não aplicável', 'Anualmente', etc.)
    for each security activity contained in the DataFrame.

    Parameters:
        df (pd.DataFrame): DataFrame with activity columns and values such as
                           'Diariamente', 'Semanalmente', 'Mensalmente', etc.

    Returns:
        pd.DataFrame: Aggregated table with frequency counts.
    """
    # Copy to avoid modifying the original DataFrame
    description_df = df.copy()

    # Adjust column names if they are in the format "column [x]"
    description_df.columns = [c.split("[")[1].replace("]", "")
                              if "[" in c else c for c in description_df.columns]
    description_df = description_df.rename(
        {"Revisão de Código": "Práticas de Segurança"}, axis=1)

    # Create frequency count table
    frequency_description = pd.DataFrame()
    for col in description_df.columns:
        counts = description_df[col].value_counts()
        frequency_description[col] = counts

    # Organize final format (activities as rows)
    frequency_description = frequency_description.fillna(0).astype(int).T

    # Reorder main columns
    frequency_description = frequency_description.reindex(
        columns=order, fill_value=0)

    return frequency_description


frequency_description = generate_description_table(
    df[frequency_cols], order=frequency_map.keys())
ease_of_use_description = generate_description_table(
    df[ease_of_use_cols], order=ease_of_use_map.keys())
effectiveness_description = generate_description_table(
    df[effectiveness_cols], order=effectiveness_map.keys())


In [27]:
df[essentiality_cols[0]].unique()

array(['Concordo', 'Concordo Totalmente', 'Neutro', 'Discordo Totalmente'],
      dtype=object)

In [28]:
for coluna in frequency_cols:
    df[coluna] = df[coluna].map(frequency_map)

for coluna in effectiveness_cols:
    df[coluna] = df[coluna].map(effectiveness_map)

for coluna in ease_of_use_cols:
    df[coluna] = df[coluna].map(ease_of_use_map)

for coluna in essentiality_cols:
    df[coluna] = df[coluna].map(essentiality_map)

In [29]:
experience_col = 'Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'
dev_experience_years_col = 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?'
security_experience_years_col = 'Quantos anos de experiência você tem em funções relacionadas a Segurança de Software?'
education_level_col = 'Qual é o seu nível de educação formal concluída?'
seniority_col = 'Por favor, indique a senioridade da posição que ocupa.'
specialist_role_col = 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?'
certification_col = 'Você possui alguma certificação na área de segurança da informação? Se sim, quais são elas?'

has_security_experience_response = 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.'

specialist_roles_responses = [
    "Cryptographer",
    "Penetration tester",
    "DevOps engineer",
    "Privacy engineer",
    "Security engineer"
]

no_certification_response = ["Não possui nenhuma certificação"]


In [30]:
df_exp_direta = df[df[experience_col] == has_security_experience_response]
df_exp_indireta = df[df[experience_col] != has_security_experience_response]

In [31]:
education_level_map = {
    'Ensino Médio': 1,
    'Ensino Superior': 2,
    'Especialização': 3,
    'Estudante de Mestrado': 4,
    'Mestrado': 5,
    'Estudante de Doutorado': 6,
    'Doutorado': 7
}

seniority_map = {
    'Estagiário': 1,
    'Júnior (até 5 anos)': 2,
    'Pleno (6 a 9 anos)': 3,
    'Sênior (10+ anos)': 4
}

dev_experience_years_map = {
    'Menos de 1 ano': 1,
    'Entre 1 e 3 anos': 2,
    'Entre 4 e 5 anos': 3,
    'Entre 7 e 14 anos': 4,
    'Mais de 15 anos': 5
}

security_experience_years_map = {
    'Nenhuma': 0,
    'Menos de 1 ano': 1,
    'Entre 1 e 3 anos': 2,
    'Entre 4 e 6 anos': 3,
    'Entre 7 e 14 anos': 4,
    'Mais de 15 anos': 5
}


def direct_experience_specialist_interaction(row):
    if row['especialista'] and row['experiencia_direta']:
        return 3
    elif row['experiencia_direta'] and not row['especialista']:
        return 2
    else:
        return 1


def direct_experience_specialist_relation(row):
    return (row['especialista'] + row['experiencia_direta']) / 2


df[dev_experience_years_col] = df[dev_experience_years_col].map(
    dev_experience_years_map)

df[seniority_col] = df[seniority_col].map(seniority_map)

df[security_experience_years_col] = df[security_experience_years_col].map(
    security_experience_years_map)


In [32]:
df['especialista'] = df[specialist_role_col].apply(
    lambda x: 1 if x in specialist_roles_responses else 0)

df['experiencia_direta'] = df[experience_col].apply(
    lambda x: 1 if x == has_security_experience_response else 0)

df['certificacao'] = df[certification_col].apply(
    lambda x: 0 if x in no_certification_response else 1)

df['Qual é o seu nível de educação formal concluída?'] = df['Qual é o seu nível de educação formal concluída?'].map(
    education_level_map)

df['interacao_exp_direta_especialista'] = df.apply(
    direct_experience_specialist_interaction, axis=1)

df['relacao_exp_direta_especialista'] = df.apply(
    direct_experience_specialist_relation, axis=1)

In [33]:
df_frequencia = df[frequency_cols].copy()
df_facilidade = df[ease_of_use_cols].copy()
df_efetividade = df[effectiveness_cols].copy()
df_essencialidade = df[essentiality_cols].copy()

In [34]:
education_level = df['Qual é o seu nível de educação formal concluída?']
dev_experience_years = df['Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?']
security_experience_years = df['Quantos anos de experiência você tem em funções relacionadas a Segurança de Software?']
seniority = df['Por favor, indique a senioridade da posição que ocupa.']
direct_experience_specialist_interaction_col = df['interacao_exp_direta_especialista']
direct_experience_specialist_relation_col = df['relacao_exp_direta_especialista']
certification = df['certificacao']


## Create the Spreadsheet

In [35]:
df.shape

(83, 107)

In [36]:
import re

# Definition of direct, indirect, large, and small responses
direct_response = 1
indirect_response = 0

small_company_response = [
    "Até 9 funcionários",
    "De 10 a 49 funcionários",
    "De 50 a 99 funcionários"
]
large_company_response = [
    "De 100 a 499 funcionários",
    "De 500 a 999 funcionários",
    "Mais de 1000 funcionários"
]

specialist_response = 1
non_specialist_response = 0

# Correct column names
direct_experience_col = 'experiencia_direta'
company_size_col = 'Qual o tamanho da empresa para a qual você trabalha atualmente?'
specialist_col = 'especialista'
systems_col = 'Quais são os tipos de sistemas que você trabalha?'

# Filter direct and indirect responses
direct_responses = df[df[direct_experience_col] == direct_response].copy()
indirect_responses = df[df[direct_experience_col] == indirect_response].copy()

# Filter large and small company responses
large_companies = df[df[company_size_col].isin(large_company_response)].copy()
small_companies = df[df[company_size_col].isin(small_company_response)].copy()

specialists = df[df[specialist_col] == specialist_response].copy()
non_specialists = df[df[specialist_col] == non_specialist_response].copy()


In [ ]:
print("Number of participants in the Direct Experience group:", len(direct_responses))
print("Number of participants in the Indirect Experience group:", len(indirect_responses))
print("--" * 35)

print("Number of participants in the Big group:", len(large_companies))
print("Number of participants in the Small group:", len(small_companies))
print("--" * 35)

print("Number of participants in the Specialists group:", len(specialists))
print("Number of participants in the Non-specialists group:",
      len(non_specialists))
print("--" * 35)

print(f"Percentage of participants in the Big group: {len(large_companies)/88:.1f}")
print(f"Percentage of participants in the Small group: {len(small_companies)/88:.1f}")


Quantidade de participantes no grupo Experiência Direta: 62
Quantidade de participantes no grupo Experiência Indireta: 21
----------------------------------------------------------------------
Quantidade de participantes no grupo Big: 65
Quantidade de participantes no grupo Small: 16
----------------------------------------------------------------------
Quantidade de participantes no grupo Especialistas: 26
Quantidade de participantes no grupo Não Especialistas: 57
----------------------------------------------------------------------
Porcentagem de participantes no grupo Big: 0.7
Porcentagem de participantes no grupo Small: 0.2


In [39]:
def remove_non_applicable(dataframe):
    total_condition = np.zeros(len(dataframe), dtype=bool)

    for column in security_practices_survey_cols:
        # Create a True condition where the column value is 0
        zero_column_condition = (dataframe[column] == 0)
        # Combine conditions using OR (|): if any is True, the row will be marked for exclusion
        total_condition = total_condition | zero_column_condition

    # Filter the DataFrame to keep only rows where the TOTAL condition is False
    # (i.e., NONE of the columns has value 0)
    return dataframe[~total_condition]


In [40]:
"""diretas = remove_non_aplicable(diretas)
indiretas = remove_non_aplicable(indiretas)

# Filtra as respostas de empresas grandes e pequenas
grandes = remove_non_aplicable(grandes)
pequenas = remove_non_aplicable(pequenas)

especialistas = remove_non_aplicable(especialistas)
nao_especialistas = remove_non_aplicable(nao_especialistas)"""

'diretas = remove_non_aplicable(diretas)\nindiretas = remove_non_aplicable(indiretas)\n\n# Filtra as respostas de empresas grandes e pequenas\ngrandes = remove_non_aplicable(grandes)\npequenas = remove_non_aplicable(pequenas)\n\nespecialistas = remove_non_aplicable(especialistas)\nnao_especialistas = remove_non_aplicable(nao_especialistas)'

In [41]:
results = []

# Function that calculates means, p-value, effect size and returns the results for a pair of groups
def calculate_data(group1, group2, column):
    group1 = group1[column][group1[column] != 0]
    group2 = group2[column][group2[column] != 0]
    group1_score = group1.mean()
    group2_score = group2.mean()

    stat, p_value = stats.mannwhitneyu(
        group1.dropna(), group2.dropna(), alternative='greater')

    # cliffs delta
    effect_size = cliffs_delta(group1, group2)

    return group1_score, group2_score, p_value, effect_size


# Function that calculates the distribution of responses on the Likert scale
# and the weighted mean for each column
def calculate_likert_distribution(group, column):

    mapped_group = group[column]
    total_responses = mapped_group.value_counts().sort_index()
    weighted_mean = sum(
        total_responses.index *
        (total_responses / total_responses.sum())
    )
    return total_responses.to_dict(), weighted_mean


In [42]:
def detect_practice_and_attribute(text: str):
    """
    Detects the text inside brackets and identifies whether it contains
    the substrings 'facilidade' or 'efetividade'.

    Returns:
        - 'practice': value inside brackets (or None if it does not exist)
        - 'attribute': 'facilidade', 'efetividade' or None
    """
    # --- 1. Detect practice (text inside brackets) ---
    match = re.search(r'\[(.*?)\]', text)
    practice = match.group(1).strip() if match else None

    # --- 2. Detect attribute ---
    attribute = None
    if 'facilidade' in text.lower():
        attribute = 'facilidade'
    elif 'efetividade' in text.lower():
        attribute = 'efetividade'
    elif 'frequência' in text.lower():
        attribute = 'frequência'
    elif 'essencial' in text.lower():
        attribute = 'essencialidade'

    return practice, attribute


In [43]:
# Loop that iterates over each survey column, performs the analyses,
# and stores the results
for survey_col in security_practices_survey_cols:
    practice, attribute = detect_practice_and_attribute(survey_col)

    total_responses, weighted_mean = calculate_likert_distribution(df, survey_col)

    direct_score, indirect_score, p_value_direct_indirect, effect_size_direct_indirect = calculate_data(
        direct_responses, indirect_responses, survey_col
    )
    
    big_score, small_score, p_value_big_small, effect_size_big_small = calculate_data(
        large_companies, small_companies, survey_col
    )
    
    specialist_score, non_specialist_score, p_value_specialist_non_specialist, effect_size_specialist_non_specialist = calculate_data(
        specialists, non_specialists, survey_col
    )

    # Create a dictionary with all computed results and append it to the results list
    row = {
        'Prática': practice,
        'Atributo': attribute,
        'Likert Média': weighted_mean,
        'Média direto': direct_score,
        'Média indireto': indirect_score,
        'P-value Direct/Indirect': p_value_direct_indirect,
        'Effect Size Direct/Indirect': effect_size_direct_indirect,

        'Média Grande': big_score,
        'Média Pequena': small_score,
        'P-value Big/Small': p_value_big_small,
        'Effect Size Big/Small': effect_size_big_small,

        'Média Especialista': specialist_score,
        'Média Não-Especialista': non_specialist_score,
        'P-value Specialist/Non-Specialist': p_value_specialist_non_specialist,
        'Effect Size Specialist/Non-Specialist': effect_size_specialist_non_specialist,
    }

    results.append(row)


In [44]:
results

[{'Prática': 'Revisão de Código',
  'Atributo': 'facilidade',
  'Likert Média': 3.0843373493975905,
  'Média direto': np.float64(3.5964912280701755),
  'Média indireto': np.float64(3.4),
  'P-value Direct/Indirect': np.float64(0.37264630631684476),
  'Effect Size Direct/Indirect': (0.05380116959064327, 'negligible'),
  'Média Grande': np.float64(3.618181818181818),
  'Média Pequena': np.float64(3.533333333333333),
  'P-value Big/Small': np.float64(0.33502499443876116),
  'Effect Size Big/Small': (0.0703030303030303, 'negligible'),
  'Média Especialista': np.float64(3.090909090909091),
  'Média Não-Especialista': np.float64(3.76),
  'P-value Specialist/Non-Specialist': np.float64(0.9920700764609476),
  'Effect Size Specialist/Non-Specialist': (-0.34363636363636363, 'medium')},
 {'Prática': 'Modelagem de Ameaças',
  'Atributo': 'facilidade',
  'Likert Média': 2.4578313253012047,
  'Média direto': np.float64(3.2830188679245285),
  'Média indireto': np.float64(3.0),
  'P-value Direct/Indir

In [45]:
analysis_results_path = Path('results/RQ2')
analysis_results_path.mkdir(exist_ok=True)

analysis_df = pd.DataFrame(results)
analysis_df.to_excel(
    analysis_results_path / 'analysis_results_raw.xlsx',
    index=False
)



## Formating table

In [ ]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import os

# --- Paths ---
input_path = results_path / "analysis_results_raw.xlsx"
output_path = results_path / "analysis_results_formatted.xlsx"

# --- Load base file ---
# Assuming that 'resultado_nao_formatado.xlsx' contains the data with correct headers
df = pd.read_excel(input_path)

print("Columns detected in the input file:", list(df.columns))

# --- Create new worksheet *without* initial Pandas header ---
writer = pd.ExcelWriter(output_path, engine='openpyxl')
pd.DataFrame().to_excel(writer, sheet_name="Análise", index=False)
writer.close()

# --- Load worksheet for formatting with Openpyxl ---
workbook = openpyxl.load_workbook(output_path)
worksheet = workbook["Análise"]

# --- Header definitions ---
main_headers = [
    'Prática', 'Atributo', 'Likert',
    'Direta vs. Indireta',
    'Grande vs. Pequena',
    'Especialista vs. Não especialista'
]

# Subcolumns: standard 4 subcolumns per comparison (Mean, Mean, p-value, d)
# Final column organization (total 15 columns):
headers = [
    'Prática',
    'Atributo',
    'Likert Média',
    # Direta vs Indireta
    'Média direto',
    'Média indireto',
    'P-value Direct/Indirect',
    'Effect Size Direct/Indirect',
    # Grande vs Pequena
    'Média Grande',
    'Média Pequena',
    'P-value Big/Small',
    'Effect Size Big/Small',
    # Especialista vs Não-Especialista
    'Média Especialista',
    'Média Não-Especialista',
    'P-value Specialist/Non-Specialist',
    'Effect Size Specialist/Non-Specialist'
]

# Merge mapping (start_col, end_col, start_row, end_row)
merge_cells = [
    (1, 1, 1, 2),   # Prática
    (2, 2, 1, 2),   # Atributo
    (3, 3, 1, 2),   # Likert
    (4, 7, 1, 1),   # Direta vs Indireta
    (8, 11, 1, 1),  # Grande vs Pequena
    (12, 15, 1, 1)  # Especialista vs Não especialista
]

# --- Styles ---
header_font = Font(bold=True)
header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
center_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
thin_border = Border(
    left=Side(style="thin"),
    right=Side(style="thin"),
    top=Side(style="thin"),
    bottom=Side(style="thin")
)

# --- Build headers in rows 1 and 2 ---
for start_col, end_col, start_row, end_row in merge_cells:
    worksheet.merge_cells(
        start_row=start_row,
        start_column=start_col,
        end_row=end_row,
        end_column=end_col
    )

# Insert row 1 headers
column_index = 1
for header in main_headers:
    cell = worksheet.cell(row=1, column=column_index, value=header)
    cell.alignment = center_alignment
    cell.font = header_font
    cell.fill = header_fill

    if header == 'Likert':
        column_index += 1
    elif header in (
        'Direta vs. Indireta',
        'Grande vs. Pequena',
        'Especialista vs. Não especialista'
    ):
        column_index += 4
    else:
        column_index += 1

# Insert row 2 headers (subcolumns)
# Column 3 (Likert Média) is vertically merged, so subheaders start at column 4
group_sub_headers = ['Média', 'Média', 'p-value', 'd']
full_sub_headers = group_sub_headers * 3
current_column = 4

for sub_header in full_sub_headers:
    cell = worksheet.cell(row=2, column=current_column, value=sub_header)
    cell.alignment = center_alignment
    cell.font = Font(bold=True)
    cell.fill = header_fill
    current_column += 1

# Apply border and alignment to header rows
for row in worksheet.iter_rows(min_row=1, max_row=2, min_col=1, max_col=15):
    for cell in row:
        cell.border = thin_border
        cell.alignment = center_alignment

# --- Write body ---
current_row = 3
for pratica, group in df.groupby("Prática"):
    num_rows = len(group)
    start_row = current_row
    end_row = current_row + num_rows - 1

    worksheet.merge_cells(
        start_row=start_row,
        start_column=1,
        end_row=end_row,
        end_column=1
    )

    pratica_cell = worksheet.cell(row=start_row, column=1, value=pratica)
    pratica_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    pratica_cell.font = Font(bold=True)
    pratica_cell.border = thin_border

    for i, (_, row) in enumerate(group.iterrows()):
        row_index = start_row + i

        for col, header in enumerate(headers[1:], start=2):
            value = row.get(header)

            if value is None or (isinstance(value, float) and pd.isna(value)):
                display_value = "-"
            elif isinstance(value, str) and '(' in value:
                value = eval(value)

                if value[0] > 0:
                    display_value = '(+)'
                elif value[0] < 0:
                    display_value = '(-)'
                elif value[0] == 0:
                    display_value = '(+)'

                if value[1] == 'large':
                    display_value += '***'
                elif value[1] == 'medium':
                    display_value += '**'
                elif value[1] == 'small':
                    display_value += '*'
                elif value[1] == 'negligible':
                    pass

            elif isinstance(value, float):
                display_value = f"{value:.2f}"
            else:
                display_value = value

            cell = worksheet.cell(row=row_index, column=col, value=display_value)
            cell.alignment = center_alignment
            cell.border = thin_border

    current_row = end_row + 1

# Auto-adjust column widths
for col in range(1, 16):
    col_letter = get_column_letter(col)
    width = 10 if col > 2 else 30
    worksheet.column_dimensions[col_letter].width = width

# Save file
workbook.save(output_path)
print(f"File formatted with a two-line header and saved at: {output_path}")


Colunas detectadas no arquivo de entrada: ['Prática', 'Atributo', 'Likert Média', 'Média direto', 'Média indireto', 'P-value Direct/Indirect', 'Effect Size Direct/Indirect', 'Média Grande', 'Média Pequena', 'P-value Big/Small', 'Effect Size Big/Small', 'Média Especialista', 'Média Não-Especialista', 'P-value Specialist/Non-Specialist', 'Effect Size Specialist/Non-Specialist']
Arquivo formatado com cabeçalho de duas linhas e salvo em: results\RQ2\analysis_results_formatted.xlsx


In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Font

# 1. Color and Style Configuration

# Color when Mean 1 > Mean 2 (Dark Gray)
GRAY_DARK_FILL = PatternFill(
    start_color="A9A9A9",
    end_color="A9A9A9",
    fill_type="solid"
)

# Color when Mean 2 > Mean 1 (Light Gray)
GRAY_LIGHT_FILL = PatternFill(
    start_color="D3D3D3",
    end_color="D3D3D3",
    fill_type="solid"
)

# Font for bold p-value
BOLD_FONT = Font(bold=True)

# 3. Column blocks to be analyzed (1-based columns, 1 = A)
# Structure: [Mean 1, Mean 2, p-value, d]
col_blocks = [
    [4, 5, 6, 7],     # Direta vs. Indireta (D, E, F, G)
    [8, 9, 10, 11],   # Grande vs. Pequena (H, I, J, K)
    [12, 13, 14, 15]  # Especialista vs. Não Especialista (L, M, N, O)
]

# 5. Function to apply formatting
def apply_formatting(xlsx_path, col_blocks):
    # Load workbook and active worksheet
    wb = load_workbook(xlsx_path)
    ws = wb.active

    # Iterate over rows, starting from row 4 (where data for the first practice begins)
    # ws.max_row is the index of the last row with data
    for row_idx in range(4, ws.max_row + 1):
        for col_block in col_blocks:
            col_mean1, col_mean2, col_p_value, col_d = col_block

            # Try to retrieve cell values.
            # Error handling is important if there are empty or non-numeric cells.
            try:
                # Getting cell values
                p_value_cell = ws.cell(row=row_idx, column=col_p_value)
                mean1_cell = ws.cell(row=row_idx, column=col_mean1)
                mean2_cell = ws.cell(row=row_idx, column=col_mean2)
                d_cell = ws.cell(row=row_idx, column=col_d)

                # Safely converting values to float.
                # If the value is None or not convertible, skip
                p_value = float(p_value_cell.value)
                mean1 = float(mean1_cell.value)
                mean2 = float(mean2_cell.value)

            except (ValueError, TypeError):
                # Ignore this row/block if values cannot be converted
                continue

            # **Formatting Logic**

            # If p-value is less than or equal to 0.05 (Significance)
            if p_value <= 0.05:

                # 1. If Mean 1 is greater than Mean 2
                if mean1 > mean2:
                    # Apply bold to p-value
                    p_value_cell.font = BOLD_FONT
                    # Fill 'd' with dark gray
                    d_cell.fill = GRAY_DARK_FILL

                # 2. If Mean 2 is greater than Mean 1
                elif mean2 > mean1:
                    # Apply bold to p-value
                    p_value_cell.font = BOLD_FONT
                    # Fill 'd' with light gray
                    d_cell.fill = GRAY_LIGHT_FILL

                # Note: If mean1 == mean2, no bold or fill is applied to 'd',
                # but it still passes the p_value <= 0.05 condition.

    # Save the modified file
    wb.save(xlsx_path)

output_xlsx_file = str(results_path / "analysis_formatted.xlsx")

print(f"Applying formatting rules to '{output_path}'...")
apply_formatting(output_path, col_blocks)

print(f"✅ Formatting completed! The file '{output_path}' was created/updated.")


# Visual example of the rule (for reference):


Aplicando as regras de formatação em 'results\RQ2\analysis_results_formatted.xlsx'...
✅ Formatação concluída! O arquivo 'results\RQ2\analysis_results_formatted.xlsx' foi criado/atualizado.
